# Merge First → Validate Data → Drop Invalid Rows

Run on **Google Colab**. Upload all 5 CSV files:

- `gee_features.csv`
- `gee_terrain_features.csv`
- `macrostrat_features.csv`
- `nasa_power_features.csv`
- `gee_terrain_failed_sites.csv`

**Quy tắc:** Full outer join 4 bảng theo **3 cột đầu** (không dựa vào số thứ tự dòng). Sau khi merge xong mới xóa các site trong danh sách failed và mọi dòng có giá trị thiếu hoặc token không hợp lệ trong bất kỳ cột nào. Nếu 1 site chỉ có trong một vài file thì dòng đó cũng bị xóa ở bước lọc.

> `Unknown` mặc định được tính là không hợp lệ. Có thể chỉnh `INVALID_TEXT_VALUES` nếu `Unknown` là dữ liệu hợp lệ của nghiên cứu. Giá trị số `0` là hợp lệ.

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files
from pathlib import Path

DATA_FILES = [
    "gee_features.csv",
    "gee_terrain_features.csv",
    "macrostrat_features.csv",
    "nasa_power_features.csv",
]
FAILED_FILE = "gee_terrain_failed_sites.csv"
OUTPUT_FILE = "merged_earth_features_clean.csv"
REJECTED_FILE = "merged_rejected_sites.csv"
REPORT_FILE = "merge_quality_report.csv"

# Adjust the list below to match your dataset conventions.
INVALID_TEXT_VALUES = {
    "", "n/a", "na", "nan", "null", "none", "unknown", "missing",
    "no data", "nodata", "not available", "-9999", "-99999",
}

print("Upload all 5 CSV files simultaneously:")
uploaded = files.upload()
missing_files = [name for name in DATA_FILES + [FAILED_FILE] if not Path(name).is_file()]
if missing_files:
    raise FileNotFoundError(f"Missing files: {missing_files}")
print("Received all 5 files.")

In [ ]:
# Read all files keeping sentinel strings intact for post-merge validation.
tables = {}
for filename in DATA_FILES:
    df = pd.read_csv(filename, low_memory=False, keep_default_na=False)
    if df.shape[1] < 4:
        raise ValueError(f"{filename}: must have at least 3 key columns and 1 data column")
    tables[filename] = df

key_cols = list(tables[DATA_FILES[0]].columns[:3])
for filename, df in tables.items():
    if list(df.columns[:3]) != key_cols:
        raise ValueError(
            f"First 3 columns in {filename} do not match the first file. "
            f"Expected: {key_cols}; found: {list(df.columns[:3])}"
        )
    if df.duplicated(subset=key_cols).any():
        example = df.loc[df.duplicated(subset=key_cols, keep=False), key_cols].head(5)
        raise ValueError(f"{filename} has duplicate site keys. Example:\n{example}")

failed = pd.read_csv(FAILED_FILE, low_memory=False, keep_default_na=False)
if list(failed.columns[:3]) != key_cols:
    raise ValueError(f"First 3 columns of {FAILED_FILE} do not match: {key_cols}")

print("Merge keys:", key_cols)
for filename, df in tables.items():
    print(f"{filename}: {len(df):,} rows, {len(df.columns)} columns")
print(f"{FAILED_FILE}: {len(failed):,} rows")

In [ ]:
# STEP 1: MERGE FIRST — do not drop NA or invalid rows yet.
# Outer join preserves every site appearing in any input file.
# Prefix duplicate data column names across files to prevent overwriting.
merged = None
source_markers = []
used_feature_names = set()

for index, filename in enumerate(DATA_FILES, start=1):
    df = tables[filename].copy()
    feature_cols = list(df.columns[3:])
    rename = {}
    for col in feature_cols:
        if col in used_feature_names:
            new_name = f"{Path(filename).stem}__{col}"
            while new_name in used_feature_names:
                new_name = "extra__" + new_name
            rename[col] = new_name
            used_feature_names.add(new_name)
        else:
            used_feature_names.add(col)
    df = df.rename(columns=rename)

    marker = f"__present_{index}__"
    df[marker] = True
    source_markers.append(marker)
    if merged is None:
        merged = df
    else:
        merged = merged.merge(df, how="outer", on=key_cols, sort=False, validate="one_to_one")

print(f"After MERGE (unfiltered): {len(merged):,} rows, {len(merged.columns) - len(source_markers)} cột thực tế")
display(merged.head())

In [ ]:
# STEP 2: VALIDATE on the MERGED table.
# Reject if: site is in failed_sites, missing in >=1 file, or contains any invalid cell.

# Match failed sites across all 3 key columns; do not drop prior to merging.
failed_keys = pd.MultiIndex.from_frame(failed[key_cols].drop_duplicates())
merged_keys = pd.MultiIndex.from_frame(merged[key_cols])
in_failed_list = pd.Series(merged_keys.isin(failed_keys), index=merged.index)

missing_source = ~merged[source_markers].fillna(False).astype(bool).all(axis=1)
real_cols = [c for c in merged.columns if c not in source_markers]

# Do not treat 0 or False as invalid. Strings are trimmed and matched case-insensitively.
invalid_by_column = pd.DataFrame(index=merged.index)
for col in real_cols:
    s = merged[col]
    invalid = s.isna()
    invalid |= s.astype("string").str.strip().str.lower().isin(INVALID_TEXT_VALUES).fillna(False)
    if pd.api.types.is_numeric_dtype(s):
        invalid |= pd.Series(~np.isfinite(pd.to_numeric(s, errors="coerce")), index=s.index)
    invalid_by_column[col] = invalid.fillna(False)

invalid_cell = invalid_by_column.any(axis=1)
reject = in_failed_list | missing_source | invalid_cell

# Rejection reasons (for auditing).
rejected = merged.loc[reject, real_cols].copy()
rejected["reason_failed_sites"] = in_failed_list.loc[reject].to_numpy()
rejected["reason_missing_from_source"] = missing_source.loc[reject].to_numpy()
rejected["reason_invalid_cell"] = invalid_cell.loc[reject].to_numpy()
rejected["invalid_columns"] = invalid_by_column.loc[reject].apply(
    lambda row: ", ".join(row.index[row]), axis=1
)

clean = merged.loc[~reject, real_cols].reset_index(drop=True)
print(f"After merge:               {len(merged):,} rows")
print(f"In failed_sites:           {int(in_failed_list.sum()):,} rows")
print(f"Missing in >=1 file:       {int(missing_source.sum()):,} rows")
print(f"Has invalid data cell:     {int(invalid_cell.sum()):,} rows")
print(f"Total removed (deduped):   {int(reject.sum()):,} rows")
print(f"REMAINING:                 {len(clean):,} rows")
print("Note: a single row may trigger multiple rejection reasons.")
display(clean.head())

In [ ]:
# STEP 3: INSPECT RESULTS & DOWNLOAD.
assert len(clean) + len(rejected) == len(merged)
assert not clean.isna().any().any()
assert not clean.duplicated(subset=key_cols).any()

report = pd.DataFrame([
    {"metric": "input_rows_" + fn, "count": len(tables[fn])} for fn in DATA_FILES
] + [
    {"metric": "rows_after_outer_merge", "count": len(merged)},
    {"metric": "rows_in_failed_site_list", "count": int(in_failed_list.sum())},
    {"metric": "rows_missing_from_one_or_more_sources", "count": int(missing_source.sum())},
    {"metric": "rows_with_invalid_cells", "count": int(invalid_cell.sum())},
    {"metric": "rows_rejected_unique", "count": int(reject.sum())},
    {"metric": "rows_final_clean", "count": len(clean)},
])

clean.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
rejected.to_csv(REJECTED_FILE, index=False, encoding="utf-8-sig")
report.to_csv(REPORT_FILE, index=False, encoding="utf-8-sig")

for name in [OUTPUT_FILE, REJECTED_FILE, REPORT_FILE]:
    files.download(name)
print("Exported 3 files: clean merged dataset, rejected rows + reasons, summary report.")

### Important Notes

- **Gộp trước, lọc sau**, đúng trình tự yêu cầu. `gee_terrain_failed_sites.csv` chỉ là danh sách site cần loại, không phải bảng đặc trưng để ghép.
- Dùng **full outer join theo 3 khóa**, thay vì ghép theo vị trí dòng: tránh làm sai site khi một file thiếu dòng.
- Dòng có bất kỳ thuộc tính thiếu hoặc `Unknown` sẽ bị loại (*complete-case filtering*). Điều này có thể làm giảm mạnh dữ liệu; nếu cần giữ lại hoặc nội suy thiếu, phải thay đổi quy tắc.
- If duplicate 3-key combinations exist within a single file, the notebook halts to avoid Cartesian join duplication.
- Very large files may exceed Colab RAM during a full outer merge; switch to chunking/SQL/DuckDB if needed.